# Notebook 01 — Análise Exploratória dos Dados (EDA)**TCC II — Predição de Churn de Clientes com Machine Learning e IA Explicável**Lucas de Jesus Mota Ferreira — FT/UNICAMP---**Referência na monografia:** Seção 3.3.1 (Análise Exploratória dos Dados)**Objetivo deste notebook:**1. Carregar a base IBM Telco Customer Churn2. Diagnosticar estrutura, tipos, nulos e duplicatas3. Caracterizar a variável-alvo e confirmar o desbalanceamento (~26,5%)4. Explorar distribuições e relações bivariadas5. Gerar hipóteses iniciais sobre fatores de evasão (validadas depois via SHAP/LIME)**Saídas geradas:** figuras em `outputs/figuras/` e tabelas em `outputs/tabelas/`

## 1. Configuração do ambiente

In [ ]:
import osimport warningsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snswarnings.filterwarnings('ignore')# Semente global (Seção 3.1 — reprodutibilidade)RANDOM_STATE = 42np.random.seed(RANDOM_STATE)# Caminhos do projetoBASE_DIR = os.path.dirname(os.getcwd())DATA_DIR = os.path.join(BASE_DIR, 'data')FIG_DIR  = os.path.join(BASE_DIR, 'outputs', 'figuras')TAB_DIR  = os.path.join(BASE_DIR, 'outputs', 'tabelas')for d in (FIG_DIR, TAB_DIR):    os.makedirs(d, exist_ok=True)# Padrão visual das figuras (consistente em todo o TCC)sns.set_theme(style='whitegrid', context='notebook')plt.rcParams['figure.figsize'] = (10, 5)plt.rcParams['figure.dpi'] = 110plt.rcParams['savefig.dpi'] = 300plt.rcParams['savefig.bbox'] = 'tight'plt.rcParams['font.size'] = 11plt.rcParams['axes.titlesize'] = 13# Paleta fixa: verde = permaneceu, vermelho = evadiuPAL = {'Não': '#2E86AB', 'Sim': '#C73E1D'}def salvar_fig(nome):    """Salva a figura atual em PNG (300 dpi) na pasta de figuras."""    caminho = os.path.join(FIG_DIR, f'{nome}.png')    plt.savefig(caminho)    print(f'Figura salva: {caminho}')print('Ambiente configurado.')

## 2. Carregamento da baseA base pode ser lida de duas formas: a partir do arquivo local em `data/` (recomendado) oudiretamente do repositório público da IBM. O trecho abaixo tenta o arquivo local primeiro.

In [ ]:
CAMINHO_LOCAL = os.path.join(DATA_DIR, 'telco_churn.csv')URL_IBM = ('https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/'           'master/data/Telco-Customer-Churn.csv')if os.path.exists(CAMINHO_LOCAL):    df = pd.read_csv(CAMINHO_LOCAL)    print('Base carregada do arquivo local.')else:    df = pd.read_csv(URL_IBM)    os.makedirs(DATA_DIR, exist_ok=True)    df.to_csv(CAMINHO_LOCAL, index=False)    print('Base baixada da fonte pública e salva localmente.')print(f'Registros: {df.shape[0]:,} | Variáveis: {df.shape[1]}')df.head()

## 3. Diagnóstico estruturalConfirmação dos números declarados na Seção 3.2.1: 7.043 registros e 21 variáveis.

In [ ]:
print('=' * 70)print('TIPOS DE DADOS')print('=' * 70)print(df.dtypes)print()print('=' * 70)print('CARDINALIDADE POR VARIÁVEL')print('=' * 70)for col in df.columns:    print(f'{col:<20} {df[col].nunique():>5} valores únicos')

In [ ]:
print(f'Valores nulos declarados: {df.isnull().sum().sum()}')print(f'Registros duplicados: {df.duplicated().sum()}')print(f'IDs de cliente duplicados: {df["customerID"].duplicated().sum()}')

### 3.1 A particularidade da variável `TotalCharges`Conforme documentado na Seção 3.3.2, `TotalCharges` é carregada como texto por conterespaços em branco em registros de clientes recém-adquiridos. A verificação abaixo confirmaesse comportamento — o tratamento efetivo ocorre no Notebook 02.

In [ ]:
print(f'Tipo original de TotalCharges: {df["TotalCharges"].dtype}')# Conversão diagnóstica (não persistida neste notebook)total_num = pd.to_numeric(df['TotalCharges'], errors='coerce')n_ausentes = total_num.isnull().sum()print(f'Registros com TotalCharges não numérico: {n_ausentes} '      f'({n_ausentes / len(df) * 100:.2f}% da base)')# Perfil desses clientes: confirmação da hipótese semântica (tenure = 0)if n_ausentes > 0:    print()    print('Perfil dos registros afetados:')    display(df.loc[total_num.isnull(), ['tenure', 'MonthlyCharges', 'Contract', 'Churn']])

## 4. Variável-alvo: distribuição e desbalanceamentoValidação empírica da proporção declarada na Seção 3.2.1 (aproximadamente 26,5% de evasores).

In [ ]:
# Rótulos em português, alinhados à padronização terminológica da monografiadf['Evasao'] = df['Churn'].map({'Yes': 'Sim', 'No': 'Não'})df['Evasao_num'] = df['Churn'].map({'Yes': 1, 'No': 0})contagem = df['Evasao'].value_counts()proporcao = df['Evasao'].value_counts(normalize=True) * 100resumo_alvo = pd.DataFrame({    'Frequência': contagem,    'Percentual (%)': proporcao.round(2)})display(resumo_alvo)razao = contagem['Não'] / contagem['Sim']print(f'Razão de desbalanceamento: {razao:.2f} : 1')resumo_alvo.to_csv(os.path.join(TAB_DIR, 'tab_distribuicao_alvo.csv'))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))sns.countplot(data=df, x='Evasao', hue='Evasao', palette=PAL, legend=False,              order=['Não', 'Sim'], ax=ax)ax.set_title('Distribuição da variável-alvo (evasão de clientes)')ax.set_xlabel('Cliente evadiu')ax.set_ylabel('Número de clientes')total = len(df)for p in ax.patches:    altura = p.get_height()    ax.annotate(f'{int(altura):,}\n({altura / total * 100:.1f}%)',                (p.get_x() + p.get_width() / 2, altura),                ha='center', va='bottom', fontsize=11)ax.set_ylim(0, contagem.max() * 1.15)plt.tight_layout()salvar_fig('eda_distribuicao_alvo')plt.show()

## 5. Variáveis numéricasAnálise das três variáveis contínuas: `tenure`, `MonthlyCharges` e `TotalCharges`.Serve de base para a decisão de padronização descrita na Seção 3.3.4.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')NUMERICAS = ['tenure', 'MonthlyCharges', 'TotalCharges']desc = df[NUMERICAS].describe().Tdesc['amplitude'] = desc['max'] - desc['min']display(desc.round(2))desc.round(2).to_csv(os.path.join(TAB_DIR, 'tab_descritivas_numericas.csv'))print('\nAs escalas divergem substancialmente — justificativa empírica para a padronização.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))titulos = {    'tenure': 'Tempo de permanência (meses)',    'MonthlyCharges': 'Cobrança mensal (R$)',    'TotalCharges': 'Cobrança acumulada (R$)'}for ax, col in zip(axes, NUMERICAS):    sns.histplot(data=df, x=col, hue='Evasao', hue_order=['Não', 'Sim'],                 palette=PAL, bins=30, kde=True, alpha=0.55, ax=ax)    ax.set_title(titulos[col])    ax.set_xlabel('')    ax.set_ylabel('Frequência')plt.tight_layout()salvar_fig('eda_distribuicoes_numericas')plt.show()

In [ ]:
# Boxplots: comparação direta entre evasores e não-evasoresfig, axes = plt.subplots(1, 3, figsize=(16, 4.5))for ax, col in zip(axes, NUMERICAS):    sns.boxplot(data=df, x='Evasao', y=col, hue='Evasao', order=['Não', 'Sim'],                palette=PAL, legend=False, ax=ax)    ax.set_title(titulos[col])    ax.set_xlabel('Cliente evadiu')    ax.set_ylabel('')plt.tight_layout()salvar_fig('eda_boxplots_numericas')plt.show()# Comparação de médias e medianas por grupocomp = df.groupby('Evasao')[NUMERICAS].agg(['mean', 'median']).round(2)display(comp)comp.to_csv(os.path.join(TAB_DIR, 'tab_comparacao_grupos.csv'))

## 6. Variáveis categóricas e taxa de evasãoCálculo da taxa de evasão por categoria. Esta é a etapa que produz as **hipóteses iniciais**mencionadas na Seção 3.3.1 — hipóteses que serão posteriormente confrontadas com osresultados do SHAP no Notebook 04.

In [ ]:
CATEGORICAS = [c for c in df.columns               if df[c].dtype == 'object'               and c not in ('customerID', 'Churn', 'Evasao')]print(f'Variáveis categóricas analisadas: {len(CATEGORICAS)}')print(CATEGORICAS)

In [ ]:
linhas = []for col in CATEGORICAS:    tab = df.groupby(col, observed=True)['Evasao_num'].agg(['count', 'mean'])    for categoria, linha in tab.iterrows():        linhas.append({            'Variável': col,            'Categoria': categoria,            'N': int(linha['count']),            'Taxa de evasão (%)': round(linha['mean'] * 100, 2)        })taxas = pd.DataFrame(linhas).sort_values('Taxa de evasão (%)', ascending=False)taxas.to_csv(os.path.join(TAB_DIR, 'tab_taxa_evasao_categorias.csv'), index=False)print('MAIORES TAXAS DE EVASÃO POR CATEGORIA')display(taxas.head(12).reset_index(drop=True))print('\nMENORES TAXAS DE EVASÃO POR CATEGORIA')display(taxas.tail(8).reset_index(drop=True))

In [ ]:
# Painel com as seis variáveis categóricas mais discriminantesamplitude = (taxas.groupby('Variável')['Taxa de evasão (%)']             .agg(lambda s: s.max() - s.min())             .sort_values(ascending=False))top6 = amplitude.head(6).index.tolist()print('Variáveis com maior poder discriminante (amplitude da taxa de evasão):')display(amplitude.head(6).round(2))fig, axes = plt.subplots(2, 3, figsize=(17, 9))for ax, col in zip(axes.ravel(), top6):    sub = (df.groupby(col, observed=True)['Evasao_num'].mean() * 100).sort_values()    sns.barplot(x=sub.values, y=sub.index, color='#C73E1D', alpha=0.85, ax=ax)    ax.set_title(col)    ax.set_xlabel('Taxa de evasão (%)')    ax.set_ylabel('')    ax.axvline(df['Evasao_num'].mean() * 100, color='#333333',               linestyle='--', linewidth=1.2)    for i, v in enumerate(sub.values):        ax.text(v + 0.6, i, f'{v:.1f}%', va='center', fontsize=9)plt.suptitle('Taxa de evasão por categoria (linha tracejada = média da base)', y=1.01)plt.tight_layout()salvar_fig('eda_taxa_evasao_categoricas')plt.show()

## 7. Correlações entre variáveis numéricas e a evasão

In [ ]:
cols_corr = NUMERICAS + ['SeniorCitizen', 'Evasao_num']matriz = df[cols_corr].corr()fig, ax = plt.subplots(figsize=(8, 6.5))mascara = np.triu(np.ones_like(matriz, dtype=bool), k=1)sns.heatmap(matriz, mask=mascara, annot=True, fmt='.2f', cmap='RdBu_r',            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,            cbar_kws={'shrink': 0.8}, ax=ax)ax.set_title('Correlação de Pearson entre variáveis numéricas e evasão')plt.tight_layout()salvar_fig('eda_matriz_correlacao')plt.show()print('Correlações com a variável-alvo:')display(matriz['Evasao_num'].drop('Evasao_num').sort_values(ascending=False).round(3))

## 8. Análise focal: tenure e tipo de contratoCruzamento das duas variáveis que a literatura correlata aponta como as mais determinantes(Rasool et al., 2025). Serve de referência para a validação posterior dos resultados de SHAP.

In [ ]:
faixas = [0, 6, 12, 24, 48, 72]rotulos = ['0-6', '7-12', '13-24', '25-48', '49-72']df['faixa_tenure'] = pd.cut(df['tenure'], bins=faixas, labels=rotulos,                            include_lowest=True)pivot = (df.pivot_table(index='faixa_tenure', columns='Contract',                        values='Evasao_num', aggfunc='mean', observed=True) * 100)fig, ax = plt.subplots(figsize=(9, 5.5))sns.heatmap(pivot, annot=True, fmt='.1f', cmap='Reds', linewidths=0.5,            cbar_kws={'label': 'Taxa de evasão (%)'}, ax=ax)ax.set_title('Taxa de evasão por faixa de permanência e tipo de contrato')ax.set_xlabel('Tipo de contrato')ax.set_ylabel('Tempo de permanência (meses)')plt.tight_layout()salvar_fig('eda_heatmap_tenure_contrato')plt.show()pivot.round(2).to_csv(os.path.join(TAB_DIR, 'tab_evasao_tenure_contrato.csv'))

## 9. Síntese: hipóteses iniciaisRegistro estruturado das hipóteses levantadas na EDA. Cada uma será confrontada com osvalores SHAP no Notebook 04, cumprindo o papel informativo descrito na Seção 3.3.1.

In [ ]:
hipoteses = pd.DataFrame([    {'ID': 'H1', 'Hipótese': 'Contratos mensais elevam substancialmente o risco de evasão'},    {'ID': 'H2', 'Hipótese': 'Clientes com baixo tempo de permanência evadem mais'},    {'ID': 'H3', 'Hipótese': 'Ausência de suporte técnico contratado eleva o risco'},    {'ID': 'H4', 'Hipótese': 'Ausência de serviços de segurança online eleva o risco'},    {'ID': 'H5', 'Hipótese': 'Fibra óptica associa-se a maior evasão que DSL'},    {'ID': 'H6', 'Hipótese': 'Pagamento por cheque eletrônico associa-se a maior evasão'},    {'ID': 'H7', 'Hipótese': 'Cobranças mensais mais altas associam-se a maior evasão'},])display(hipoteses)hipoteses.to_csv(os.path.join(TAB_DIR, 'tab_hipoteses_eda.csv'), index=False)print()print('=' * 70)print('RESUMO DA ETAPA DE EDA')print('=' * 70)print(f'Registros: {len(df):,}')print(f'Variáveis preditoras: {df.shape[1] - 5}')print(f'Taxa de evasão: {df["Evasao_num"].mean() * 100:.2f}%')print(f'Razão de desbalanceamento: {razao:.2f} : 1')print(f'Registros com TotalCharges ausente: {df["TotalCharges"].isnull().sum()}')print(f'Variáveis categóricas a codificar: {len(CATEGORICAS)}')print('\nPróxima etapa: Notebook 02 — Pré-processamento')